# CUDA Programming in PyTorch: A Practical Guide

This notebook accompanies the 10-chapter book. Each chapter section contains
runnable code examples. Chapters 3–5 require a CUDA GPU and an installed
C++ compiler. Chapters 7–10 can be explored with PyTorch's CPU fallbacks.

**Prerequisites**
```
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
pip install triton          # Chapter 7 (Linux only)
pip install tensorboard     # Chapter 8 profiler traces
```

In [ ]:
import torch
import sys

print(f"PyTorch  : {torch.__version__}")
print(f"Python   : {sys.version.split()[0]}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA ver : {torch.version.cuda}")
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## Chapter 1 — Why Custom CUDA in PyTorch?

PyTorch dispatches every operator to a pre-written CUDA kernel.  
Chaining ops like `relu(x + bias)` launches two kernels with two HBM round-trips.
A fused custom kernel does the same work in one pass.

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
x    = torch.randn(4096, 4096, device=device)
bias = torch.randn(4096,       device=device)

# Standard PyTorch: 2 kernel launches, 2 HBM round-trips
def unfused(x, bias):
    return torch.relu(x + bias)

# Count distinct operators dispatched
import torch.fx as fx

def count_ops(fn, *args):
    """Symbolically trace and count aten ops."""
    try:
        traced = fx.symbolic_trace(fn)
        ops = [n.target for n in traced.graph.nodes if n.op == 'call_function']
        return ops
    except Exception:
        return ["tracing not available for this function"]

# Show the ops that make up unfused()
import torch.nn as nn

class UnfusedModel(nn.Module):
    def forward(self, x, bias):
        return torch.relu(x + bias)

ops = count_ops(UnfusedModel(), x, bias)
print("Operators in unfused relu(x + bias):")
for op in ops:
    print(f"  {op}")

print("\nA fused kernel collapses all of these into a single kernel launch.")

---
## Chapter 2 — Environment & Project Setup

Verify your environment matches a known-good configuration before writing any kernel code.

In [ ]:
import torch
import subprocess, shutil, sys

print("=== Environment check ===")
print(f"torch          : {torch.__version__}")
print(f"torch.cuda     : {torch.version.cuda}")

# Check for nvcc
nvcc = shutil.which('nvcc')
if nvcc:
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    nvcc_ver = result.stdout.splitlines()[-1].strip()
    print(f"nvcc           : {nvcc_ver}")
else:
    print("nvcc           : NOT FOUND (install CUDA Toolkit)")

# Check C++ compiler
cxx = shutil.which('c++') or shutil.which('g++')
print(f"c++ compiler   : {cxx or 'NOT FOUND'}")

# Show include paths that extensions will use
from torch.utils.cpp_extension import include_paths
print("\nTorch include paths:")
for p in include_paths(cuda=True)[:3]:  # show first 3
    print(f"  {p}")

In [ ]:
# ---------------------------------------------------------------
# Minimal inline extension (no .cu file needed — for CPU demo)
# ---------------------------------------------------------------
from torch.utils.cpp_extension import load_inline

# A trivial C++ op that squares a float tensor element-wise (CPU only)
cpp_src = """
#include <torch/extension.h>
torch::Tensor square_cpu(torch::Tensor x) {
    return x * x;
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("square", &square_cpu, "Square elements (CPU)");
}
"""

ext = load_inline(
    name='square_ext',
    cpp_sources=[cpp_src],
    verbose=False,
)

x = torch.tensor([1.0, 2.0, 3.0, 4.0])
out = ext.square(x)
print("Input : ", x.tolist())
print("Output:", out.tolist())
assert torch.allclose(out, x**2), "Mismatch!"
print("✓ inline extension works")

---
## Chapter 3 — Your First CUDA Kernel

The minimal pattern: `__global__` kernel → C++ launcher → pybind11 binding → Python call.

> **Requires CUDA GPU.** The cell detects and skips gracefully on CPU-only machines.

In [ ]:
import torch
from torch.utils.cpp_extension import load_inline

if not torch.cuda.is_available():
    print("SKIP: No CUDA GPU detected.")
else:
    cuda_src = """
    #include <cuda_runtime.h>
    #include <torch/extension.h>

    __global__ void relu_kernel(const float* __restrict__ x,
                                float* __restrict__ out, int N) {
        int i = blockIdx.x * blockDim.x + threadIdx.x;
        if (i < N) out[i] = x[i] > 0.f ? x[i] : 0.f;
    }

    torch::Tensor relu_cuda(torch::Tensor x) {
        TORCH_CHECK(x.is_cuda(),       "x must be a CUDA tensor");
        TORCH_CHECK(x.is_contiguous(), "x must be contiguous");
        TORCH_CHECK(x.scalar_type() == torch::kFloat, "float32 only");
        auto out = torch::empty_like(x);
        int N = x.numel();
        const int threads = 256;
        const int blocks  = (N + threads - 1) / threads;
        relu_kernel<<<blocks, threads>>>(
            x.data_ptr<float>(), out.data_ptr<float>(), N);
        return out;
    }
    """

    cpp_src = """
    torch::Tensor relu_cuda(torch::Tensor x);
    PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
        m.def("relu", &relu_cuda, "Custom ReLU (CUDA)");
    }
    """

    relu_ext = load_inline(
        name='relu_ext',
        cpp_sources=[cpp_src],
        cuda_sources=[cuda_src],
        extra_cuda_cflags=['-O3'],
        verbose=False,
    )

    x = torch.randn(1_000_000, device='cuda')
    out = relu_ext.relu(x)

    # Correctness check against PyTorch reference
    ref = torch.relu(x)
    assert torch.allclose(out, ref), "Mismatch!"
    print(f"✓ Custom ReLU correct on {x.numel():,} elements")
    print(f"  max abs error: {(out - ref).abs().max().item():.2e}")

---
## Chapter 4 — Memory & Data Layout

Key concepts: contiguous vs non-contiguous tensors, memory coalescing, stride-aware access.

In [ ]:
import torch

# Demonstrate contiguous vs non-contiguous
x = torch.randn(4, 6)
print("Original tensor:")
print(f"  shape  : {x.shape}")
print(f"  strides: {x.stride()}   ← row-major: move 6 floats to next row")
print(f"  contiguous: {x.is_contiguous()}")

xt = x.T
print("\nTransposed tensor:")
print(f"  shape  : {xt.shape}")
print(f"  strides: {xt.stride()}   ← non-contiguous: stride[0]=1, stride[1]=6")
print(f"  contiguous: {xt.is_contiguous()}")

# Making contiguous is a copy
xt_c = xt.contiguous()
print(f"\nAfter .contiguous():")
print(f"  strides: {xt_c.stride()}")
print(f"  contiguous: {xt_c.is_contiguous()}")
print(f"  same storage? {xt.data_ptr() == xt_c.data_ptr()}")

In [ ]:
import torch

if not torch.cuda.is_available():
    print("SKIP: CUDA required for coalescing benchmark.")
else:
    import time

    N = 4096
    x = torch.randn(N, N, device='cuda')

    def bench(fn, repeats=200):
        for _ in range(10): fn()  # warm-up
        torch.cuda.synchronize()
        start = torch.cuda.Event(enable_timing=True)
        end   = torch.cuda.Event(enable_timing=True)
        start.record()
        for _ in range(repeats): fn()
        end.record()
        torch.cuda.synchronize()
        return start.elapsed_time(end) / repeats  # ms

    # Row-sum: coalesced (threads in a warp read adjacent columns)
    t_row = bench(lambda: x.sum(dim=1))

    # Column-sum: non-coalesced pattern for the kernel internally
    t_col = bench(lambda: x.sum(dim=0))

    print(f"Row sum   (coalesced-friendly): {t_row:.3f} ms")
    print(f"Col sum   (transpose pattern) : {t_col:.3f} ms")
    print(f"Ratio     : {t_col/t_row:.2f}x")
    print("\nNote: PyTorch's internals handle this well — the gap is")
    print("much larger in naive custom kernels.")

---
## Chapter 5 — Custom C++/CUDA Extensions

Multi-dtype dispatch with `AT_DISPATCH_FLOATING_TYPES_AND_HALF` and TORCH_LIBRARY registration.

In [ ]:
import torch
from torch.utils.cpp_extension import load_inline

if not torch.cuda.is_available():
    print("SKIP: CUDA required.")
else:
    cuda_src = """
    #include <cuda_runtime.h>
    #include <torch/extension.h>

    template <typename scalar_t>
    __global__ void gelu_kernel(const scalar_t* __restrict__ x,
                                scalar_t* __restrict__ out, int N) {
        int i = blockIdx.x * blockDim.x + threadIdx.x;
        if (i < N) {
            float v  = static_cast<float>(x[i]);
            const float c = 0.7978845608f;
            float t = c * (v + 0.044715f * v * v * v);
            out[i] = static_cast<scalar_t>(0.5f * v * (1.f + tanhf(t)));
        }
    }

    torch::Tensor gelu_cuda(torch::Tensor x) {
        TORCH_CHECK(x.is_cuda());
        auto out = torch::empty_like(x);
        int N = x.numel();
        int threads = 256, blocks = (N + 255) / 256;
        AT_DISPATCH_FLOATING_TYPES_AND_HALF(x.scalar_type(), "gelu_cuda", [&]() {
            gelu_kernel<scalar_t><<<blocks, threads>>>(
                x.data_ptr<scalar_t>(), out.data_ptr<scalar_t>(), N);
        });
        return out;
    }
    """

    cpp_src = """
    torch::Tensor gelu_cuda(torch::Tensor x);
    PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
        m.def("gelu", &gelu_cuda, "Custom GELU (float32 + float16)");
    }
    """

    gelu_ext = load_inline(
        name='gelu_ext',
        cpp_sources=[cpp_src],
        cuda_sources=[cuda_src],
        extra_cuda_cflags=['-O3'],
        verbose=False,
    )

    for dtype in [torch.float32, torch.float16]:
        x   = torch.randn(2**20, device='cuda', dtype=dtype)
        out = gelu_ext.gelu(x)
        ref = torch.nn.functional.gelu(x)
        tol = 1e-3 if dtype == torch.float16 else 1e-5
        ok  = torch.allclose(out.float(), ref.float(), atol=tol)
        print(f"dtype={dtype}: max_err={( out.float()-ref.float()).abs().max():.2e}  {'✓' if ok else '✗'}")

---
## Chapter 6 — Autograd Integration

Wrap a custom forward+backward in `torch.autograd.Function` and verify gradients.

In [ ]:
import torch

# Pure-Python GELU function + analytic backward for demonstration.
# In production this would call CUDA kernels instead.

class GeluFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x: torch.Tensor) -> torch.Tensor:
        ctx.save_for_backward(x)
        # Tanh GELU approximation
        c = 0.7978845608
        return 0.5 * x * (1.0 + torch.tanh(c * (x + 0.044715 * x**3)))

    @staticmethod
    def backward(ctx, grad_out: torch.Tensor) -> torch.Tensor:
        (x,) = ctx.saved_tensors
        c   = 0.7978845608
        t   = torch.tanh(c * (x + 0.044715 * x**3))
        dgelu = 0.5 * (1.0 + t) + 0.5 * x * (1.0 - t**2) * c * (1.0 + 3*0.044715 * x**2)
        return grad_out * dgelu


def fast_gelu(x):
    return GeluFunction.apply(x)


# Correctness check
x = torch.randn(128, dtype=torch.float64, requires_grad=True)
out = fast_gelu(x)
ref = torch.nn.functional.gelu(x)
print(f"Forward max_err : {(out - ref).abs().max().item():.2e}")

# Gradient check (uses finite differences — needs float64)
x_gc = torch.randn(16, dtype=torch.float64, requires_grad=True)
ok = torch.autograd.gradcheck(fast_gelu, (x_gc,), eps=1e-5, atol=1e-4)
print(f"gradcheck       : {'PASSED ✓' if ok else 'FAILED ✗'}")

In [ ]:
import torch
import torch.nn as nn

# Plug the custom Function into a real module
class CustomGeluMLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in,     d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        return self.fc2(fast_gelu(self.fc1(x)))


device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = CustomGeluMLP(64, 256, 10).to(device)
optim  = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for step in range(50):
    x      = torch.randn(32, 64, device=device)
    target = torch.randint(10, (32,), device=device)
    loss   = nn.functional.cross_entropy(model(x), target)
    loss.backward()
    optim.step()
    optim.zero_grad()
    losses.append(loss.item())

print(f"Training with custom GELU on {device}")
print(f"  initial loss : {losses[0]:.4f}")
print(f"  final   loss : {losses[-1]:.4f}")
print("  ✓ gradients flow through custom Function")

---
## Chapter 7 — Triton: Python-Level Kernels

Triton lets you write tile-level GPU kernels in Python. No `threadIdx`, no warp-level intrinsics — Triton handles scheduling.

> **Requires `triton` package** (`pip install triton`) and a CUDA GPU.

In [ ]:
try:
    import triton
    import triton.language as tl
    print(f"Triton version: {triton.__version__}")
    HAS_TRITON = True
except ImportError:
    print("triton not installed — run: pip install triton")
    HAS_TRITON = False

In [ ]:
if not (HAS_TRITON and torch.cuda.is_available()):
    print("SKIP: Triton + CUDA required.")
else:
    import triton
    import triton.language as tl
    import torch

    @triton.jit
    def softmax_kernel(
        X_ptr, Out_ptr,
        stride_row,
        N: tl.constexpr,
        BLOCK: tl.constexpr,
    ):
        row  = tl.program_id(0)
        cols = tl.arange(0, BLOCK)
        mask = cols < N
        x    = tl.load(X_ptr + row * stride_row + cols,
                       mask=mask, other=-float('inf'))
        x    = x - tl.max(x, axis=0)
        x    = tl.exp(x)
        x    = x / tl.sum(x, axis=0)
        tl.store(Out_ptr + row * stride_row + cols, x, mask=mask)


    def triton_softmax(x: torch.Tensor) -> torch.Tensor:
        rows, cols = x.shape
        out   = torch.empty_like(x)
        BLOCK = triton.next_power_of_2(cols)
        softmax_kernel[(rows,)](
            x, out,
            stride_row=x.stride(0),
            N=cols,
            BLOCK=BLOCK,
            num_warps=4 if BLOCK <= 2048 else 8,
        )
        return out

    x   = torch.randn(512, 1024, device='cuda')
    out = triton_softmax(x)
    ref = torch.softmax(x, dim=1)
    print(f"Triton softmax max_err: {(out - ref).abs().max().item():.2e}")
    assert torch.allclose(out, ref, atol=1e-5)
    print("✓ Triton softmax matches torch.softmax")

In [ ]:
if not (HAS_TRITON and torch.cuda.is_available()):
    print("SKIP.")
else:
    import torch

    def bench_ms(fn, repeats=200, warmup=25):
        for _ in range(warmup): fn()
        torch.cuda.synchronize()
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record()
        for _ in range(repeats): fn()
        e.record()
        torch.cuda.synchronize()
        return s.elapsed_time(e) / repeats

    x = torch.randn(2048, 2048, device='cuda')
    t_triton = bench_ms(lambda: triton_softmax(x))
    t_torch  = bench_ms(lambda: torch.softmax(x, dim=1))

    print(f"Triton softmax : {t_triton:.3f} ms")
    print(f"torch.softmax  : {t_torch:.3f}  ms")
    print(f"Ratio          : {t_triton/t_torch:.2f}x")

---
## Chapter 8 — Profiling & Benchmarking

The reusable benchmark harness from the book, plus PyTorch Profiler.

In [ ]:
import torch

def benchmark(fn, *args, warmup=25, repeats=200):
    """
    GPU-side benchmark using CUDA Events.
    Returns median elapsed time in milliseconds.
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA required")

    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)
    times = []

    for _ in range(repeats):
        start.record()
        fn(*args)
        end.record()
        torch.cuda.synchronize()
        times.append(start.elapsed_time(end))

    times.sort()
    return times[len(times) // 2]  # median


if torch.cuda.is_available():
    x = torch.randn(4096, 4096, device='cuda', dtype=torch.float16)

    t_relu  = benchmark(torch.relu, x)
    t_gelu  = benchmark(torch.nn.functional.gelu, x)
    t_silu  = benchmark(torch.nn.functional.silu, x)

    bytes_rw = 2 * x.numel() * x.element_size()

    print(f"Activation benchmark on {x.shape} float16 tensor")
    print(f"{'Op':<12} {'ms':>8} {'TB/s':>8}")
    print("-" * 32)
    for name, t in [("relu", t_relu), ("gelu", t_gelu), ("silu", t_silu)]:
        bw = bytes_rw / (t * 1e-3) / 1e12
        print(f"{name:<12} {t:>8.3f} {bw:>8.2f}")
else:
    print("SKIP: CUDA required for benchmark.")

In [ ]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model  = nn.Sequential(
    nn.Linear(512, 2048), nn.GELU(),
    nn.Linear(2048, 512),
).to(device)

x = torch.randn(128, 512, device=device)

activities = [ProfilerActivity.CPU]
if torch.cuda.is_available():
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    with_stack=False,
) as prof:
    for _ in range(5):
        with record_function("forward"):
            out = model(x)
        with record_function("backward"):
            out.sum().backward()

sort_by = 'cuda_time_total' if torch.cuda.is_available() else 'cpu_time_total'
print(prof.key_averages().table(sort_by=sort_by, row_limit=10))

---
## Chapter 9 — Multi-GPU & CUDA Streams

CUDA streams for compute/transfer overlap; DistributedDataParallel (DDP) for multi-GPU training.

In [ ]:
import torch

if not torch.cuda.is_available():
    print("SKIP: CUDA required.")
else:
    # Two streams — demonstrate concurrent execution
    compute_stream  = torch.cuda.Stream()
    transfer_stream = torch.cuda.Stream()

    x_cpu = torch.randn(8192, 8192, pin_memory=True)  # pinned = async-copyable

    # Transfer on transfer_stream
    with torch.cuda.stream(transfer_stream):
        x_gpu = x_cpu.to('cuda', non_blocking=True)

    # Compute waits for transfer to finish
    compute_stream.wait_stream(transfer_stream)

    with torch.cuda.stream(compute_stream):
        result = torch.relu(x_gpu)

    torch.cuda.synchronize()
    print(f"Result shape: {result.shape}, sum: {result.sum().item():.2f}")
    print("✓ Compute/transfer overlap using CUDA streams")

In [ ]:
import torch

n = torch.cuda.device_count()
print(f"Available CUDA devices: {n}")
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name} — {p.total_memory / 1e9:.1f} GB VRAM  "
          f"(sm_{p.major}{p.minor}, {p.multi_processor_count} SMs)")

if n >= 2:
    # Quick tensor-parallel split example
    x  = torch.randn(1024, 512)
    x0 = x[:512].cuda(0)  # first half on GPU 0
    x1 = x[512:].cuda(1)  # second half on GPU 1
    print(f"\nTensor split across 2 GPUs:")
    print(f"  GPU 0 shard: {x0.shape}")
    print(f"  GPU 1 shard: {x1.shape}")
elif n == 1:
    print("\nOnly 1 GPU — DDP example omitted (requires 2+).")
else:
    print("\nNo GPU — multi-GPU examples require CUDA hardware.")

In [ ]:
# DDP training loop — reference code (not executed here;
# launch with: torchrun --nproc_per_node=NUM_GPUS train.py)

DDP_TEMPLATE = '''
import os, torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

def setup(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

def train(rank, world_size):
    setup(rank, world_size)
    torch.cuda.set_device(rank)

    model = MyModel().cuda(rank)
    model = DDP(model, device_ids=[rank])  # wraps model; backward does all-reduce

    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    for batch in dataloader:
        out  = model(batch["x"].cuda(rank))
        loss = out.sum()
        loss.backward()   # all-reduce gradients across GPUs
        optimizer.step(); optimizer.zero_grad()

    dist.destroy_process_group()

# Entry point:
# torch.multiprocessing.spawn(train, args=(NUM_GPUS,), nprocs=NUM_GPUS)
'''

print("DDP template (save as train.py and launch with torchrun):")
print(DDP_TEMPLATE)

---
## Chapter 10 — Packaging & Production

Abstract (meta) functions for `torch.compile` compatibility; multi-arch `setup.py`; CPU fallback dispatcher.

In [ ]:
import torch
from torch.library import Library, impl

# Define a custom op namespace
lib = Library("demo_ops", "DEF")
lib.define("gelu(Tensor x) -> Tensor")

# CPU implementation (used as fallback and for testing)
@impl(lib, "gelu", "CPU")
def _gelu_cpu(x):
    return torch.nn.functional.gelu(x)

# CUDA implementation (if available)
if torch.cuda.is_available():
    @impl(lib, "gelu", "CUDA")
    def _gelu_cuda(x):
        # In production: call the compiled CUDA extension here
        return torch.nn.functional.gelu(x)  # placeholder

# Abstract implementation for torch.compile / FakeTensor tracing
from torch.library import impl_abstract
@impl_abstract("demo_ops::gelu")
def _gelu_abstract(x):
    return torch.empty_like(x)  # same shape & dtype, no actual compute


# Usage
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(8, device=device)
out = torch.ops.demo_ops.gelu(x)
print(f"torch.ops.demo_ops.gelu: {out.shape} on {out.device}")
print(f"max_err vs reference   : {(out - torch.nn.functional.gelu(x)).abs().max().item():.2e}")

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def model_fn(x):
    # Uses our registered op — torch.compile will trace through it
    return torch.ops.demo_ops.gelu(x) * 2.0

# Compile the model — the abstract fn tells the compiler the output shape
compiled = torch.compile(model_fn)

x   = torch.randn(512, device=device)
out = compiled(x)
print(f"torch.compile output shape: {out.shape}")
print("✓ Custom op is torch.compile-compatible")

In [ ]:
SETUP_PY_TEMPLATE = '''
# setup.py for a production CUDA extension wheel
from setuptools import setup, find_packages
from torch.utils.cpp_extension import BuildExtension, CUDAExtension

# Target common modern architectures
ARCHS = ["70", "80", "89", "90"]
gencode = []
for arch in ARCHS:
    gencode += [f"-gencode=arch=compute_{arch},code=sm_{arch}"]
gencode += ["-gencode=arch=compute_90,code=compute_90"]  # PTX fallback

setup(
    name="my_cuda_ops",
    version="0.1.0",
    packages=find_packages(),
    ext_modules=[CUDAExtension(
        "my_cuda_ops._C",
        sources=["csrc/ops.cpp", "csrc/gelu_kernel.cu"],
        extra_compile_args={
            "cxx":  ["-O3", "-std=c++17"],
            "nvcc": ["-O3", "--use_fast_math"] + gencode,
        },
    )],
    cmdclass={"build_ext": BuildExtension},
)
# Build wheel: python setup.py bdist_wheel
# Install:     pip install dist/*.whl
'''

print(SETUP_PY_TEMPLATE)

---
## Appendix — Quick Reference

| Chapter | Core concept | Key API |
|---------|-------------|--------|
| 1 — Why custom CUDA | Framework ceiling, HBM round-trips | PyTorch dispatch |
| 2 — Setup | CUDA toolkit ↔ PyTorch ABI | `load()`, `CUDAExtension`, nvcc |
| 3 — First kernel | `__global__`, `data_ptr`, grid/block | `load_inline`, pybind11 |
| 4 — Memory | Strides, coalescing, `__shared__` | `PackedTensorAccessor` |
| 5 — Custom extensions | Multi-dtype dispatch, shape checks | `AT_DISPATCH`, `TORCH_LIBRARY` |
| 6 — Autograd | Custom forward + backward | `torch.autograd.Function`, `gradcheck` |
| 7 — Triton | Tile-level Python kernels | `@triton.jit`, `tl.load`/`tl.store` |
| 8 — Profiling | CUDA Events, roofline | `torch.profiler`, `ncu` |
| 9 — Multi-GPU | Streams, device guards, DDP | `torch.cuda.Stream`, `DDP` |
| 10 — Production | Multi-arch, compile compat | `impl_abstract`, `setup.py`, wheels |